In [1]:
import pandas as pd
from datetime import datetime
import plotly.express as px

In [2]:
# https://www.data.go.kr/data/15049591/fileData.do <= 표준산업분류코드 데이터
def std_convert(str):
    return f"0{str}" if len(str) == 1 else str
STD_TOBZ_M_df = pd.read_csv("dataset/고용노동부_표준산업분류코드.csv")
STD_TOBZ_M_df['고용업종코드(중분류)'] = STD_TOBZ_M_df['고용업종코드(중분류)'].astype(int).astype(str)
STD_TOBZ_M_df['고용업종코드(중분류)'] = STD_TOBZ_M_df['고용업종코드(중분류)'].apply(lambda x : std_convert(x))
STD_TOBZ_M_df['고용업종코드(소분류)'] = STD_TOBZ_M_df['고용업종코드(소분류)'].astype(str)

STD_TOBZ_M_dic = dict(zip(STD_TOBZ_M_df['고용업종코드(중분류)'], STD_TOBZ_M_df['고용업종명(중분류)'])) # 표준산업분류 - 중분류 dict
STD_TOBZ_S_dic = dict(zip(STD_TOBZ_M_df['고용업종코드(소분류)'], STD_TOBZ_M_df['고용업종명(소분류)'])) # 표준산업분류 - 소분류 dict

In [3]:
# km data preprocessing
def exc_age(age):
    age = age.split(".")[-1].split("대")[0]
    try:
        return "60+" if int(age) >= 60 else age
    except:
        return age
km_data = pd.read_csv("dataset/국민카드.csv", encoding='cp949')
km_data = km_data[~km_data['표준산업세세분류코드'].isna()]
km_data['표준산업세세분류코드'] = km_data['표준산업세세분류코드'].astype(int).astype(str)
km_data['표준산업세세분류코드'] = km_data['표준산업세세분류코드'].apply(lambda x : x[:3])
km_data['STD_TOBZ_M_C'] = km_data['표준산업세세분류코드'].apply(lambda x : x[:2])
km_data['기준년월'] = pd.to_datetime(km_data['기준년월'], format='%Y%m')
km_data['연령대'] = km_data['연령대'].apply(exc_age)
km_data['company'] = '국민'
km_data.columns = ['date', 'city_l', 'city_m', 'city_s', 'sex', 'age', 'code_s_name', 'code_s', 'cost', 'case', 'client', 'code_m', 'company']
km_data = km_data[['date', 'city_l', 'city_m', 'city_s', 'sex', 'age', 'company', 'code_m', 'code_s', 'cost', 'case']]
km_data['case'] = km_data['case'].astype(int)

# sh data preprocessing
city_l_dic = {'전남':'전라남도', '경기':'경기도', '서울':'서울특별시'}
sex_dic = {'F':'여성', 'M':'남성'}
sh_data = pd.read_csv("dataset/신한카드.csv")
sh_data['MGPO_NM_HOM'] = sh_data['MGPO_NM_HOM'].map(city_l_dic)
sh_data['SEX_C'] = sh_data['SEX_C'].map(sex_dic)
sh_data['SL_CRI_YM'] = pd.to_datetime(sh_data['SL_CRI_YM'], format='%Y%m')
sh_data['STD_TOBZ_M_C'] = sh_data['STD_TOBZ_M_C'].astype(str)
sh_data['company'] = '신한'
sh_data = sh_data[['SL_CRI_YM', 'MGPO_NM_HOM', 'SGG_NM_HOM', 'NASD_NM_HOM', 'SEX_C', 'AGE_G5_C', 'company', 'STD_TOBZ_M_C', 'STD_TOBZ_L_C','AMT', 'CNT']]
sh_data.columns = ['date', 'city_l', 'city_m', 'city_s', 'sex', 'age', 'company', 'code_m', 'code_s', 'cost', 'case']
sh_data['case'] = sh_data['case'].apply(lambda x : int(x) if x.isdigit() else 0)

# ss data preprocessing
def exc_age(age):
    return "60+" if int(age) >= 60 else age      
ss_data = pd.read_csv("dataset/삼성카드.csv")
ss_data['표준업종분류코드_중분류'] = ss_data['표준업종분류코드_중분류'].astype(str)
ss_data['거주지_광역시도'] = ss_data['거주지_광역시도'].map(city_l_dic)
ss_data['성별'] = ss_data['성별'].map(sex_dic)
ss_data['이용기준년월'] = pd.to_datetime(ss_data['이용기준년월'], format='%Y%m')
ss_data['연령대'] = ss_data['연령대'].apply(exc_age)
ss_data['company'] = '삼성'
ss_data = ss_data[['이용기준년월', '거주지_광역시도', '거주지_시군구', '거주지_행정동', '성별', '연령대', 'company', '표준업종분류코드_중분류', '표준업종분류코드_소분류','이용금액', '이용건수']]
ss_data.columns = ['date', 'city_l', 'city_m', 'city_s', 'sex', 'age', 'company', 'code_m', 'code_s', 'cost', 'case']
ss_data['case'] = ss_data['case'].astype(int)

# data concat
con_df = pd.concat([km_data, sh_data, ss_data]).reset_index(drop=True) # 3사 데이터 통합
con_df = con_df.drop_duplicates()
jg_df = con_df[con_df['city_m']=='중구'].reset_index(drop=True) # 중구 data만 filtering
ddc_df = con_df[con_df['city_m']=='동두천시'].reset_index(drop=True) # 동두천시 data만 filtering
nj_df = con_df[con_df['city_m']=='나주시'].reset_index(drop=True) # 나주시 data만 filtering

In [7]:
jg_df.to_csv("jg_df.csv", index=False)
ddc_df.to_csv("ddc_df.csv", index=False)
nj_df.to_csv("nj_df.csv", index=False)

In [5]:
# data concat
jg_df

,date,city_l,city_m,city_s,sex,age,company,code_m,code_s,cost,case
0,2023-04-01,서울특별시,중구,다산동,여성,50,국민,56,561,24370810,683
1,2023-05-01,서울특별시,중구,약수동,남성,60+,국민,59,591,20000,3
2,2023-08-01,서울특별시,중구,신당5동,남성,60+,국민,47,478,129200,3
3,2023-03-01,서울특별시,중구,명동,여성,40,국민,56,561,407250,27
4,2023-11-01,서울특별시,중구,다산동,남성,60+,국민,47,474,696940,8
...,...,...,...,...,...,...,...,...,...,...,...
302084,2022-12-01,서울특별시,중구,명동,남성,30,삼성,90,902,32000,3
302085,2023-07-01,서울특별시,중구,다산동,남성,40,삼성,90,902,29000,2
302086,2023-01-01,서울특별시,중구,소공동,여성,30,삼성,90,902,31000,2
302087,2023-03-01,서울특별시,중구,중림동,남성,40,삼성,90,902,137000,13


In [20]:
jg_df[jg_df['date']=='2023-04-01']

,date,city_l,city_m,city_s,sex,age,company,code_m,code_s,cost,case
0,2023-04-01,서울특별시,중구,다산동,여성,50,국민,56,561,24370810,683
18,2023-04-01,서울특별시,중구,신당동,여성,60+,국민,47,479,8726128,160
19,2023-04-01,서울특별시,중구,다산동,여성,10,국민,49,492,457400,44
24,2023-04-01,서울특별시,중구,필동,남성,20,국민,47,476,785780,7
26,2023-04-01,서울특별시,중구,다산동,남성,20,국민,75,751,133630,18
...,...,...,...,...,...,...,...,...,...,...,...
302001,2023-04-01,서울특별시,중구,신당5동,남성,20,삼성,47,476,136000,4
302017,2023-04-01,서울특별시,중구,소공동,여성,50,삼성,47,473,203000,9
302032,2023-04-01,서울특별시,중구,회현동,남성,60+,삼성,47,473,386000,8
302049,2023-04-01,서울특별시,중구,약수동,남성,30,삼성,46,466,6000,2


In [ ]:
px.box(jg_df, x="city_s", y="cost")

In [ ]:
jg_df['company']

In [32]:
px.density_heatmap(jg_df, x="company", y="city_s")

In [29]:
px.density_heatmap(jg_df, x="city_s", y="age", facet_row="sex", facet_col="company")

In [ ]:
import plotly.graph_objects as go
fig = go.Figure(data=[go.Surface(z=jg_df['city_s'], x=jg_df['age'], y=jg_df['cost'])])
fig.update_layout(title='Mt Bruno Elevation', autosize=False,
                  width=500, height=500,
                  margin=dict(l=65, r=50, b=65, t=90))
fig.show()

In [7]:
jg_df[jg_df['date']=='2023-04-01']

,date,city_l,city_m,city_s,sex,age,company,code_m,code_s,cost,case
0,2023-04-01,서울특별시,중구,다산동,여성,50,국민,56,561,24370810,683
18,2023-04-01,서울특별시,중구,신당동,여성,60+,국민,47,479,8726128,160
19,2023-04-01,서울특별시,중구,다산동,여성,10,국민,49,492,457400,44
24,2023-04-01,서울특별시,중구,필동,남성,20,국민,47,476,785780,7
26,2023-04-01,서울특별시,중구,다산동,남성,20,국민,75,751,133630,18
...,...,...,...,...,...,...,...,...,...,...,...
302001,2023-04-01,서울특별시,중구,신당5동,남성,20,삼성,47,476,136000,4
302017,2023-04-01,서울특별시,중구,소공동,여성,50,삼성,47,473,203000,9
302032,2023-04-01,서울특별시,중구,회현동,남성,60+,삼성,47,473,386000,8
302049,2023-04-01,서울특별시,중구,약수동,남성,30,삼성,46,466,6000,2


In [7]:
codem_count = jg_df['code_m'].value_counts()
codem_count

code_m
47    92135
56    29457
49    18603
91    15428
96    15058
86    13981
61    10324
73     9263
65     8382
52     7929
85     7860
95     7384
55     6356
84     5960
75     5608
68     5134
46     4403
90     4361
51     3903
94     3098
59     2892
33     2876
45     2738
66     2159
18     2051
76     2004
62     1956
10     1805
71     1752
29     1511
35     1308
50      984
87      731
21      698
60      560
42      467
74      349
58      323
70      131
34       68
41       58
14       37
30        4
Name: count, dtype: int64

In [6]:
jg_count_dic = dict(zip(jg_df['code_m'].value_counts().index,jg_df['code_m'].value_counts().values))
jg_std = pd.DataFrame({'표준산업분류코드':sorted(jg_df['code_m'].unique())})
jg_std['표준산업분류(중)'] = jg_std['표준산업분류코드'].map(STD_TOBZ_M_dic)
jg_std['데이터건수'] = jg_std['표준산업분류코드'].map(jg_count_dic)
jg_std = jg_std.sort_values(by='데이터건수').reset_index(drop=True)
jg_std

,표준산업분류코드,표준산업분류(중),데이터건수
0,30,자동차 및 트레일러 제조업,4
1,14,의복 의복 액세서리 및 모피제품 제조업,37
2,41,종합 건설업,58
3,34,산업용 기계 및 장비 수리업,68
4,70,연구개발업,131
5,58,출판업,323
6,74,사업시설 관리 및 조경 서비스업,349
7,42,전문직별 공사업,467
8,60,방송업,560
9,21,의료용 물질 및 의약품 제조업,698


In [ ]:
jg_std

In [ ]:
fig = px.bar(
    x=jg_std['데이터건수'],
    y=jg_std['표준산업분류(중)'],
    color=jg_std['표준산업분류(중)'],
    color_continuous_scale='agsunset'
)

# 레이아웃 업데이트
fig.update_layout(
    xaxis_title='Counts',
    yaxis_title='Standard Industry',
    title='Number of Standard Industry',
    margin=dict(l=270), 
    template='plotly_dark',  # 사용할 템플릿
    font=dict(
        family='Arial, sans-serif',  # 사용할 폰트 설정
        size=12,  # 폰트 크기 설정
    ),
)
# 그림을 보여주기
fig.show()